In [1]:
# %pip install -r ../requirements.txt

# Importing the data which is present in 3 splits.
 Here we import the data present already into the 3 splits as Training data, Testing data, and Validation data and concatinate it to get a complete dataset.
- The Data is inspected by concatinating all the splits and performing duplicacy checks.
- We then calculate 3 Baseline Scores by using 10 folds cross-validation:
    - TF-IDF + Logistic Regression
    - Naive Bayes SVM (NBSVM)
    - CNN-rand

In [2]:
import datasets
class RottenTomatoesMovieReview(datasets.GeneratorBasedBuilder):
    """Cornell Rotten Tomatoes movie reviews dataset."""

    VERSION = datasets.Version("1.0.0") 

    
    def _split_generators(self, dl_manager):
        """Downloads Rotten Tomatoes sentences."""
        archive  = dl_manager.download_and_extract("https://huggingface.co/datasets/cornell-movie-review-data/resolve/main/rotten_tomatoes.zip")
        return [
            datasets.SplitGenerator(
                name=datasets.Split.TRAIN,
                gen_kwargs={"split_key": "train", "files": dl_manager.iter_archive(archive)},
            ),
            datasets.SplitGenerator(
                name=datasets.Split.VALIDATION,
                gen_kwargs={"split_key": "validation", "files": dl_manager.iter_archive(archive)},
            ),
            datasets.SplitGenerator(
                name=datasets.Split.TEST,
                gen_kwargs={"split_key": "test", "files": dl_manager.iter_archive(archive)},
            ),
        ]

m:\Coding\Convolutional Neural Networks for Sentence Classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Number of samples in the entire dataset
import pandas as pd
df_train = pd.read_parquet("hf://datasets/cornell-movie-review-data/rotten_tomatoes/train.parquet")
print(f"Number of samples in the training dataset: {len(df_train)}")
df_test = pd.read_parquet("hf://datasets/cornell-movie-review-data/rotten_tomatoes/test.parquet")
print(f"Number of samples in the test dataset: {len(df_test)}")
df_validation = pd.read_parquet("hf://datasets/cornell-movie-review-data/rotten_tomatoes/validation.parquet")
print(f"Number of samples in the validation dataset: {len(df_validation)}")
df = pd.concat([df_train, df_test, df_validation], ignore_index=True)
num_samples = len(df)
print(f"Number of samples in the entire dataset: {num_samples}")
sample_len = len(df.iloc[0]['text'])
print(f"Length of the first sample: {sample_len}")
avg_len = df['text'].apply(len).mean()
print(f"Average length of samples in the entire dataset: {avg_len}")
print("\n")
print("\n")

# Classes or Targets in the dataset
classes = df['label'].unique()
print(f"Classes in the dataset: {classes}")

# Is class balance count or bar chart
class_counts = df['label'].value_counts()
print(f"Class counts in the dataset:\n{class_counts}")

Number of samples in the training dataset: 8530
Number of samples in the test dataset: 1066
Number of samples in the validation dataset: 1066
Number of samples in the entire dataset: 10662
Length of the first sample: 177
Average length of samples in the entire dataset: 114.1591633839805




Classes in the dataset: [1 0]
Class counts in the dataset:
label
1    5331
0    5331
Name: count, dtype: int64


In [4]:
# Remove rows with missing reviews or labels
df = df.dropna(subset=["text", "label"])

# Remove duplicate rows
df = df.drop_duplicates(subset=["text", "label"])


# Keep only positive and negative reviews
df = df[df["label"].isin([0, 1])]

print(df["label"].value_counts())
print("No duplicates and missing values in the dataset.")

label
1    5331
0    5331
Name: count, dtype: int64
No duplicates and missing values in the dataset.


### TF-IDF + Logistic Regression for 10 folds CV

In [5]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


X = df["text"].fillna("").astype(str)
y = df["label"].astype(int)

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

lr_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=20000,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

scores = cross_validate(
    lr_model,
    X,
    y,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=-1
)

print("Fold accuracies:", scores["test_accuracy"])
print("Mean accuracy:", scores["test_accuracy"].mean())
print("Accuracy std:", scores["test_accuracy"].std())

print("Mean macro-F1:", scores["test_f1_macro"].mean())
print("Macro-F1 std:", scores["test_f1_macro"].std())

Fold accuracies: [0.74507966 0.76476101 0.75984991 0.76454034 0.77204503 0.77485929
 0.78236398 0.76547842 0.74577861 0.76172608]
Mean accuracy: 0.7636482325821022
Accuracy std: 0.0111089777138096
Mean macro-F1: 0.7635983316141793
Macro-F1 std: 0.0111188050761364


### Naive Bayes Support Vector Machine for 10 fold CV:

In [10]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

accuracies = []
macro_f1_scores = []

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    X_train_text = X.iloc[train_idx]
    X_test_text = X.iloc[test_idx]

    y_train = y.iloc[train_idx].to_numpy()
    y_test = y.iloc[test_idx].to_numpy()

    # Fit vocabulary on this fold's training data only
    vectorizer = CountVectorizer(
        binary=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=200000,
        dtype=np.float64
    )

    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    # Calculate Naive Bayes feature ratios from training data only
    positive_counts = np.asarray(
        X_train[y_train == 1].sum(axis=0)
    ).ravel()

    negative_counts = np.asarray(
        X_train[y_train == 0].sum(axis=0)
    ).ravel()

    alpha = 1.0

    positive_probs = positive_counts + alpha
    negative_probs = negative_counts + alpha

    positive_probs /= positive_probs.sum()
    negative_probs /= negative_probs.sum()

    r = np.log(positive_probs / negative_probs)

    # Reweight features using the ratios
    X_train_nb = X_train.multiply(r).tocsr()
    X_test_nb = X_test.multiply(r).tocsr()

    # Train the linear classifier
    model = LinearSVC(
        C=0.1,
        max_iter=5000,
        random_state=42
    )

    model.fit(X_train_nb, y_train)
    predictions = model.predict(X_test_nb)

    acc = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average="macro")

    accuracies.append(acc)
    macro_f1_scores.append(f1)

    print(f"Fold {fold}: accuracy={acc:.4f}, macro-F1={f1:.4f}")

print("\nNBSVM 10-fold results")
print(f"Mean accuracy: {np.mean(accuracies):.4f}")
print(f"Accuracy std:  {np.std(accuracies):.4f}")
print(f"Mean macro-F1: {np.mean(macro_f1_scores):.4f}")
print(f"Macro-F1 std:  {np.std(macro_f1_scores):.4f}")

Fold 1: accuracy=0.7694, macro-F1=0.7694
Fold 2: accuracy=0.7901, macro-F1=0.7900
Fold 3: accuracy=0.7777, macro-F1=0.7777
Fold 4: accuracy=0.7871, macro-F1=0.7870
Fold 5: accuracy=0.7889, macro-F1=0.7889
Fold 6: accuracy=0.7702, macro-F1=0.7701
Fold 7: accuracy=0.7795, macro-F1=0.7795
Fold 8: accuracy=0.7974, macro-F1=0.7974
Fold 9: accuracy=0.7795, macro-F1=0.7794
Fold 10: accuracy=0.8058, macro-F1=0.8057

NBSVM 10-fold results
Mean accuracy: 0.7846
Accuracy std:  0.0110
Mean macro-F1: 0.7845
Macro-F1 std:  0.0110


## Different split, the standard 80/20 split approach:
Here we split the data into 80% training and 20% testing to check if we get better or similar results.
The same 3 Baseline models are again created to compare.

In [6]:
from sklearn.model_selection import train_test_split
df = pd.concat([df_train, df_test, df_validation], ignore_index=True)
df['label'] = df['label'].astype(int)
df_new = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f"Training dataset size: {len(df_new[0])}")
print(f"Validation dataset size: {len(df_new[1])}")

Training dataset size: 8529
Validation dataset size: 2133


In [7]:
# Check for duplicates in the new training and validation datasets
train_duplicates = df_new[0].duplicated(subset=["text", "label"]).sum()
val_duplicates = df_new[1].duplicated(subset=["text", "label"]).sum()
print(f"Number of duplicates in the new training dataset: {train_duplicates}")
print(f"Number of duplicates in the new validation dataset: {val_duplicates}")

# Check for samples appearing in both training and validation datasets
train_texts = set(df_new[0]["text"])
val_texts = set(df_new[1]["text"])
common_texts = train_texts.intersection(val_texts)
print(f"Number of samples appearing in both datasets: {len(common_texts)}")

Number of duplicates in the new training dataset: 0
Number of duplicates in the new validation dataset: 0
Number of samples appearing in both datasets: 0


### TF-IDF + Logistic Regression for 80/20 split:

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Extract text and labels
X_train = df_new[0]["text"].astype(str)
y_train = df_new[0]["label"]

# 2. Initialize TF-IDF
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=20000,
    ngram_range=(1, 2)
)

# 3. Fit TF-IDF on training data and transform it
X_train_tfidf = tfidf.fit_transform(X_train)

# 4. Initialize Logistic Regression
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# 5. Train the classifier
model.fit(X_train_tfidf, y_train)

print("Training completed!")

Training completed!


In [19]:
# Testing the model on the validation dataset
X_val = df_new[1]["text"].astype(str)
y_val = df_new[1]["label"]

X = tfidf.transform(X_val)

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
model.fit(X, y_val)

print("Validation completed!")
print(f"Validation accuracy: {model.score(X, y_val):.4f}")
print(f"Validation macro-F1: {f1_score(y_val, model.predict(X), average='macro'):.4f}")

Validation completed!
Validation accuracy: 0.9939
Validation macro-F1: 0.9939


### Naive Baye's Support Vector Machine for 80/20 split:

In [18]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

X_train_text = df_new[0]["text"].astype(str)
X_test_text = df_new[1]["text"].astype(str)

y_train = df_new[0]["label"].to_numpy()
y_test = df_new[1]["label"].to_numpy()

# Fit vocabulary on training data only
vectorizer = CountVectorizer(
    binary=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=200000,
    dtype=np.float64
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# Calculate Naive Bayes feature ratios from training data only
positive_counts = np.asarray(
    X_train[y_train == 1].sum(axis=0)
).ravel()

negative_counts = np.asarray(
    X_train[y_train == 0].sum(axis=0)
).ravel()

alpha = 1.0

positive_probs = positive_counts + alpha
negative_probs = negative_counts + alpha

positive_probs /= positive_probs.sum()
negative_probs /= negative_probs.sum()

r = np.log(positive_probs / negative_probs)

# Reweight features using the ratios
X_train_nb = X_train.multiply(r).tocsr()
X_test_nb = X_test.multiply(r).tocsr()

# Train the linear classifier
model = LinearSVC(
    C=0.1,
    max_iter=5000,
    random_state=42
)

model.fit(X_train_nb, y_train)
predictions = model.predict(X_test_nb)

acc = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, average="macro")

print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1-score: {f1:.4f}")



Validation accuracy: 0.7768
Validation F1-score: 0.7768


# CNN-Rand code for the 10 fold Cross Validation
```python

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D,
    Dropout, Dense, Concatenate
)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
MAX_VOCAB = 30000
MAX_LEN = 100
EMBEDDING_DIM = 300
FILTERS = 100
BATCH_SIZE = 50
EPOCHS = 25

tf.keras.utils.set_random_seed(SEED)


def build_cnn_rand(vocab_size):
    """Kim-style CNN-rand for binary sentiment classification."""

    inputs = Input(shape=(MAX_LEN,), dtype="int32")

    # Randomly initialized embeddings; trained from scratch.
    embedded = Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        embeddings_initializer="uniform",
        trainable=True
    )(inputs)

    pooled_outputs = []

    # Three parallel convolution branches: widths 3, 4 and 5.
    for width in [3, 4, 5]:
        conv = Conv1D(
            filters=FILTERS,
            kernel_size=width,
            activation="relu",
            padding="valid",
            kernel_constraint=max_norm(3.0, axis=[0, 1])
        )(embedded)

        pooled = GlobalMaxPooling1D()(conv)
        pooled_outputs.append(pooled)

    merged = Concatenate()(pooled_outputs)
    dropped = Dropout(0.5)(merged)

    # Binary output: probability of class 1.
    outputs = Dense(
        1,
        activation="sigmoid",
        kernel_constraint=max_norm(3.0, axis=0)
    )(dropped)

    model = Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adadelta(),
        loss="binary_crossentropy",
        metrics=["accuracy"],
        jit_compile=False
      )

    return model


def make_sequences(train_text, other_texts):
    """
    Fit the tokenizer ONLY on training text.
    Transform the other sets using the same vocabulary.
    """
    tokenizer = Tokenizer(
        num_words=MAX_VOCAB,
        oov_token="<OOV>"
    )

    tokenizer.fit_on_texts(train_text)

    sequences = [
        tokenizer.texts_to_sequences(texts)
        for texts in [train_text] + list(other_texts)
    ]

    padded = [
        pad_sequences(
            seq,
            maxlen=MAX_LEN,
            padding="post",
            truncating="post"
        )
        for seq in sequences
    ]

    # Include OOV and padding indices in the embedding vocabulary.
    vocab_size = min(
        MAX_VOCAB,
        len(tokenizer.word_index) + 1
    )

    return padded, vocab_size


def fit_and_evaluate(train_text, y_train, val_text, y_val,
                     eval_text, y_eval, verbose=0):
    """
    Train with a development set for early stopping,
    then evaluate on a separate held-out set.
    """
    (X_tr, X_val, X_eval), vocab_size = make_sequences(
        train_text,
        [val_text, eval_text]
    )

    model = build_cnn_rand(vocab_size)

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    model.fit(
        X_tr,
        np.asarray(y_train, dtype=np.float32),
        validation_data=(
            X_val,
            np.asarray(y_val, dtype=np.float32)
        ),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=verbose
    )

    probabilities = model.predict(X_eval, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype(int)

    accuracy = accuracy_score(y_eval, predictions)
    macro_f1 = f1_score(y_eval, predictions, average="macro")

    return model, predictions, accuracy, macro_f1

TEXT_COL = "text"
LABEL_COL = "label"

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

# Encode labels consistently.
label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(
    df[LABEL_COL].astype(str)
)

if len(label_encoder.classes_) != 2:
    raise ValueError(
        f"Expected 2 sentiment classes, got {label_encoder.classes_}"
    )

X_all = df[TEXT_COL]

print("Total reviews:", len(df))
print("Classes:", label_encoder.classes_)
print("Class counts:", np.bincount(y_all))

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED
)

fold_results = []

for fold, (outer_train_idx, outer_test_idx) in enumerate(
    cv.split(X_all, y_all), start=1
):
    print(f"\n========== Fold {fold}/10 ==========")

    # Split only the outer training portion into train and development.
    inner_train_idx, dev_idx = train_test_split(
        outer_train_idx,
        test_size=0.10,
        random_state=SEED,
        stratify=y_all[outer_train_idx]
    )

    _, predictions, accuracy, macro_f1 = fit_and_evaluate(
        train_text=X_all.iloc[inner_train_idx].tolist(),
        y_train=y_all[inner_train_idx],
        val_text=X_all.iloc[dev_idx].tolist(),
        y_val=y_all[dev_idx],
        eval_text=X_all.iloc[outer_test_idx].tolist(),
        y_eval=y_all[outer_test_idx],
        verbose=0
    )

    fold_results.append({
        "fold": fold,
        "accuracy": accuracy,
        "macro_f1": macro_f1
    })

    print(f"Fold accuracy: {accuracy:.4f}")
    print(f"Fold macro-F1: {macro_f1:.4f}")

results_df = pd.DataFrame(fold_results)

print("\n========== 10-Fold CNN-rand Results ==========")
display(results_df)

print(f"Mean accuracy: {results_df['accuracy'].mean():.4f}")
print(f"Accuracy std:  {results_df['accuracy'].std(ddof=1):.4f}")
print(f"Mean macro-F1: {results_df['macro_f1'].mean():.4f}")
print(f"Macro-F1 std:  {results_df['macro_f1'].std(ddof=1):.4f}")

```
### Note: This code is executed on Google Collab with GPU as it might not be supported by a CPU.

# Results CNN-Rand 10 folds cross validation:
<font size=2>
Total reviews: 10662

Classes: ['0' '1']

Class counts: [5331 5331]

========== Fold 1/10 ==========

Fold accuracy: 0.5136

Fold macro-F1: 0.4412

========== Fold 2/10 ==========

Fold accuracy: 0.5108

Fold macro-F1: 0.4246


========== Fold 3/10 ==========

Fold accuracy: 0.5141

Fold macro-F1: 0.5134


========== Fold 4/10 ==========

Fold accuracy: 0.5047

Fold macro-F1: 0.4030


========== Fold 5/10 ==========

Fold accuracy: 0.5000

Fold macro-F1: 0.4831


========== Fold 6/10 ==========

Fold accuracy: 0.5263

Fold macro-F1: 0.4701


========== Fold 7/10 ==========

Fold accuracy: 0.5084

Fold macro-F1: 0.3860


========== Fold 8/10 ==========

Fold accuracy: 0.5244

Fold macro-F1: 0.5065


========== Fold 9/10 ==========

Fold accuracy: 0.5019

Fold macro-F1: 0.3635


========== Fold 10/10 ==========

Fold accuracy: 0.5000

Fold macro-F1: 0.4086


========== 10-Fold CNN-rand Results ==========
  
Mean accuracy: 0.5104

Accuracy std:  0.0094

Mean macro-F1: 0.4400

Macro-F1 std:  0.0516

</font>

# CNN-Rand code for a standard 80/20 split:
```python
from sklearn.model_selection import train_test_split

df['label'] = df['label'].astype(int)
df_new = train_test_split(df, test_size = 0.2, random_state=42, stratify=df['label'])
# print(df_new[0].shape)
# print(df_new[1].shape)


df_train_fixed = df_new[0].copy()
df_test_fixed = df_new[1].copy()

df_train_fixed[TEXT_COL] = (
    df_train_fixed[TEXT_COL].fillna("").astype(str)
)
df_test_fixed[TEXT_COL] = (
    df_test_fixed[TEXT_COL].fillna("").astype(str)
)

# Use the label encoder fitted above so class IDs are consistent.
X_train_full = df_train_fixed[TEXT_COL].tolist()
y_train_full = label_encoder.transform(
    df_train_fixed[LABEL_COL].astype(str)
)

X_test = df_test_fixed[TEXT_COL].tolist()
y_test = label_encoder.transform(
    df_test_fixed[LABEL_COL].astype(str)
)

# Reserve 10% of the training split for early stopping.
train_idx, dev_idx = train_test_split(
    np.arange(len(X_train_full)),
    test_size=0.10,
    random_state=SEED,
    stratify=y_train_full
)

model_fixed, test_predictions, test_accuracy, test_macro_f1 = (
    fit_and_evaluate(
        train_text=[X_train_full[i] for i in train_idx],
        y_train=y_train_full[train_idx],
        val_text=[X_train_full[i] for i in dev_idx],
        y_val=y_train_full[dev_idx],
        eval_text=X_test,
        y_eval=y_test,
        verbose=1
    )
)

print("\n========== Fixed 80/20 CNN-rand Results ==========")
print(f"Training split size: {len(df_train_fixed)}")
print(f"Test split size:     {len(df_test_fixed)}")
print(f"Test accuracy:       {test_accuracy:.4f}")
print(f"Test macro-F1:       {test_macro_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    labels=[0, 1],
    target_names=label_encoder.classes_.astype(str)
))
```
### Note: This code is executed on Google Collab with GPU as it might not be supported by a CPU.

# Results CNN-Rand for a standard 80/20 split:
<font size=2>

Epoch 1/25

154/154- accuracy: 0.4917 - loss: 0.6953 - val_accuracy: 0.4889 - val_loss: 0.6942

Epoch 2/25

154/154- accuracy: 0.5046 - loss: 0.6946 - val_accuracy: 0.4936 - val_loss: 0.6942

Epoch 3/25

154/154- accuracy: 0.5057 - loss: 0.6937 - val_accuracy: 0.4936 - val_loss: 0.6942

Epoch 4/25

154/154- accuracy: 0.5109 - loss: 0.6936 - val_accuracy: 0.4936 - val_loss: 0.6942

Epoch 5/25

154/154- accuracy: 0.4983 - loss: 0.6945 - val_accuracy: 0.4947 - val_loss: 0.6941

Epoch 6/25

154/154- accuracy: 0.5089 - loss: 0.6941 - val_accuracy: 0.4924 - val_loss: 0.6941

Epoch 7/25

154/154- accuracy: 0.4958 - loss: 0.6953 - val_accuracy: 0.4924 - val_loss: 0.6941

Epoch 8/25

154/154- accuracy: 0.5031 - loss: 0.6935 - val_accuracy: 0.4924 - val_loss: 0.6941

Epoch 9/25

154/154- accuracy: 0.5081 - loss: 0.6935 - val_accuracy: 0.4900 - val_loss: 0.6940

Epoch 10/25

154/154- accuracy: 0.5031 - loss: 0.6945 - val_accuracy: 0.4889 - val_loss: 0.6940

Epoch 11/25

154/154- accuracy: 0.5132 - loss: 0.6936 - val_accuracy: 0.4912 - val_loss: 0.6940

Epoch 12/25

154/154- accuracy: 0.4979 - loss: 0.6942 - val_accuracy: 0.4912 - val_loss: 0.6940

Epoch 13/25

154/154- accuracy: 0.5069 - loss: 0.6939 - val_accuracy: 0.4912 - val_loss: 0.6939

Epoch 14/25

154/154- accuracy: 0.5085 - loss: 0.6938 - val_accuracy: 0.4912 - val_loss: 0.6939

Epoch 15/25

154/154- accuracy: 0.5072 - loss: 0.6937 - val_accuracy: 0.4912 - val_loss: 0.6939

Epoch 16/25

154/154- accuracy: 0.5143 - loss: 0.6924 - val_accuracy: 0.4900 - val_loss: 0.6939

Epoch 17/25

154/154- accuracy: 0.5098 - loss: 0.6934 - val_accuracy: 0.4912 - val_loss: 0.6938

Epoch 18/25

154/154- accuracy: 0.4999 - loss: 0.6940 - val_accuracy: 0.4947 - val_loss: 0.6938

Epoch 19/25

154/154- accuracy: 0.5070 - loss: 0.6936 - val_accuracy: 0.4947 - val_loss: 0.6938

Epoch 20/25

154/154- accuracy: 0.5193 - loss: 0.6927 - val_accuracy: 0.4959 - val_loss: 0.6938

Epoch 21/25

154/154- accuracy: 0.5116 - loss: 0.6931 - val_accuracy: 0.4982 - val_loss: 0.6937

Epoch 22/25

154/154- accuracy: 0.5063 - loss: 0.6936 - val_accuracy: 0.4982 - val_loss: 0.6937

Epoch 23/25

154/154- accuracy: 0.5125 - loss: 0.6933 - val_accuracy: 0.5006 - val_loss: 0.6937

Epoch 24/25

154/154- accuracy: 0.5133 - loss: 0.6925 - val_accuracy: 0.5018 - val_loss: 0.6937

Epoch 25/25

154/154- accuracy: 0.5098 - loss: 0.6935 - val_accuracy: 0.5006 - val_loss: 0.6936

========== Fixed 80/20 CNN-rand Results ==========

Training split size: 8529

Test split size:     2133

Test accuracy:       0.5251

Test macro-F1:       0.4803

Classification report:

| | precision | recall | f1-score | support |
|---|---|---|---|---|
| 0 | 0.56 | 0.23 | 0.33 | 1067 |
| 1 | 0.52 | 0.82 | 0.63 | 1066 |
| accuracy | | | 0.53 | 2133 |
| macro avg | 0.54 | 0.53 | 0.48 | 2133 |
|weighted avg | 0.54 | 0.53 | 0.48 | 2133 |

</font>